In [1]:
import os
import json
from dotenv import load_dotenv
from sqlalchemy import create_engine, select, update
from sqlalchemy.orm import sessionmaker
from models import Comment
from filter import *

In [2]:
load_dotenv()

db_host = os.getenv('db_host')
db_user = os.getenv('db_user')
db_password = os.getenv('db_password')
db_port = os.getenv('db_port')
db_name = os.getenv('db_name')

In [3]:
DATABASE_URL = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)

In [4]:
with open('security_keywords.txt', 'r') as f:
    security_keywords = [line.strip() for line in f.readlines() if line.strip()]

print(security_keywords)

['TOCTOU', 'breach', 'code injection', 'cross site', 'ddos', 'dead lock', 'dead-lock', 'deadlock', 'deadlocks', 'denial of service', 'exploit', 'forged', 'gain access', 'infinite recursion', 'infinite-recursion', 'leak', 'malicious', 'malicious code', 'overflow', 'overrun', 'race', 'races', 'racy', 'redos', 'sql injection', 'stack overflow', 'unauthenticated', 'underflow', 'vulnerability', 'vulnerable', 'xsrf', 'xss', 'xxe', 'zip slip', 'zipslip']


In [5]:
# assumes: contains_keyword(text, security_keywords) -> (bool, list_of_keywords)
# and SQLAlchemy models: Comment, Session

BATCH_SIZE = 5_000  # tune to your RAM/IO profile

def fetch_comments_in_batches(session, batch_size=BATCH_SIZE):
    """
    Generator that yields rows of Comment in batches using keyset pagination on Comment.id.
    Respects your filters: possible_response == False and in_reply_to_id is NULL.
    """
    last_id = None
    base_filters = [
        Comment.possible_response.is_(False),
        Comment.in_reply_to_id.is_(None),
    ]

    cols = (
        Comment.id,
        Comment.body,
        Comment.author,
        Comment.diff_hunk,
        Comment.pr_id,
        Comment.owner,
        Comment.name,
        Comment.original_commit_id,
        Comment.path,
    )

    while True:
        filters = list(base_filters)
        if last_id is not None:
            filters.append(Comment.id > last_id)

        qry = (
            select(*cols)
            .where(*filters)
            .order_by(Comment.id)
            .limit(batch_size)
        )

        batch = session.execute(qry).all()
        if not batch:
            break

        # advance cursor for next page
        last_id = batch[-1][0]  # first column is Comment.id

        # yield the batch to caller
        for row in batch:
            yield row

comments = []
counter = 0
matched = 0

with Session() as session:
    for row in fetch_comments_in_batches(session):
        # unpack by position to keep it fast; row is a tuple in the same order as cols
        (
            comment_id,
            body,
            author,
            diff_hunk,
            pr_id,
            owner,
            name,
            original_commit_id,
            path,
        ) = row

        counter += 1
        if counter % 5000 == 0:
            print(f"Scanned {counter} comments...", end="\r")

        con, k = contains_keyword(body, security_keywords)
        if con and path.endswith('.java'):
            matched += 1
            comments.append({
                'id': comment_id,
                'body': body,
                'author': author,
                'diff': diff_hunk,
                'keywords': k,
                'pr_id': pr_id,
                'owner': owner,
                'name': name,
                'url': f'https://github.com/{owner}/{name}/pull/{pr_id}/files/{original_commit_id}',
                'path': path,
            })

print(f"Total scanned: {counter} | Total comments with security keywords: {matched}")


Total scanned: 113552 | Total comments with security keywords: 343


In [6]:
comments[0]

{'id': 4183230,
 'body': "This is a race-condition and wouldn't work (I believe) ... but it's actually made redundant and will never get applied since `completed` will only ever equal `observers.size()` once since `completed` atomically increases each time it's called.\n",
 'author': 'benjchristensen',
 'diff': '@@ -174,18 +161,15 @@ public Aggregator(FuncN<R> combineLatestFunction) {\n          * \n          * @param w The observer that has completed.\n          */\n-        <T> void complete(CombineObserver<R, T> w) {\n-            synchronized(lockObject) {\n-                // store that this CombineLatestObserver is completed\n-                completed.add(w);\n-                // if all CombineObservers are completed, we mark the whole thing as completed\n-                if (completed.size() == observers.size()) {\n-                    if (running.get()) {\n-                        // mark ourselves as done\n-                        observer.onCompleted();\n-                   

In [7]:
print(len(comments))

343


In [8]:
# Read from excel
import pandas as pd
df = pd.read_excel('real_dataset_with_security_keywords.xlsx')

In [9]:
df.head()

,id,body,author,diff,keywords,pr_id,owner,name,url,Sec
0,4183230,This is a race-condition and wouldn't work (I ...,benjchristensen,"@@ -174,18 +161,15 @@ public Aggregator(FuncN<...",['race'],267,ReactiveX,RxJava,https://github.com/ReactiveX/RxJava/pull/267/f...,Y
1,6992896,Would the use of `compareAndSet` patterns simp...,benjchristensen,"@@ -0,0 +1,128 @@\n+/**\n+ * Copyright 2013 Ne...",['race'],434,ReactiveX,RxJava,https://github.com/ReactiveX/RxJava/pull/434/f...,N
2,10122253,can `maxTermFrequency * maxDoc` overflow?\n,jpountz,"@@ -72,4 +83,96 @@ public void setLowFreqMinim...",['overflow'],5273,elastic,elasticsearch,https://github.com/elastic/elasticsearch/pull/...,Y
3,15079514,Why is this considered such a problem? This is...,benjchristensen,"@@ -70,7 +70,9 @@ public void request(long n) ...",['race'],1454,ReactiveX,RxJava,https://github.com/ReactiveX/RxJava/pull/1454/...,N
4,15500831,It could be overflow even if not Long.MAX_VALUE\n,benjchristensen,"@@ -108,11 +108,21 @@ void startEmitting() {\n...",['overflow'],1523,ReactiveX,RxJava,https://github.com/ReactiveX/RxJava/pull/1523/...,Y


In [10]:
ids = []

for comment in comments:
    ids.append(comment['id'])

In [11]:
# Copy to 2 new dataframes
# one where the id in df['id'] is in comments
# another where the id in df['id'] is in comments and
# Sec == 'Y'

df_java = df[df['id'].isin(ids)]
df_java_sec = df_java[df_java['Sec'] == 'Y']

In [12]:
#Print the info
df_java.info()

<class 'pandas.core.frame.DataFrame'>
Index: 343 entries, 0 to 399
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        343 non-null    int64 
 1   body      343 non-null    object
 2   author    343 non-null    object
 3   diff      343 non-null    object
 4   keywords  343 non-null    object
 5   pr_id     343 non-null    int64 
 6   owner     343 non-null    object
 7   name      343 non-null    object
 8   url       343 non-null    object
 9   Sec       343 non-null    object
dtypes: int64(2), object(8)
memory usage: 29.5+ KB


In [13]:
#Save df_java to excel
df_java.to_excel('real_dataset_java_comments.xlsx', index=False)

In [14]:
df_java_sec.info()

<class 'pandas.core.frame.DataFrame'>
Index: 190 entries, 0 to 397
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        190 non-null    int64 
 1   body      190 non-null    object
 2   author    190 non-null    object
 3   diff      190 non-null    object
 4   keywords  190 non-null    object
 5   pr_id     190 non-null    int64 
 6   owner     190 non-null    object
 7   name      190 non-null    object
 8   url       190 non-null    object
 9   Sec       190 non-null    object
dtypes: int64(2), object(8)
memory usage: 16.3+ KB


In [15]:
# Save to excel
df_java_sec.to_excel('real_dataset_java_security_comments.xlsx', index=False)

In [16]:
# get all keywords elements in df_java_sec['Keywords']
all_keywords = df_java_sec['keywords'].explode().unique()

# They are lists that where transformed to strings when saving to excel
# so turn then back to lists
k = []
for item in all_keywords:
    if isinstance(item, str):
        item = item.strip("[]").replace("'", "").split(", ")
        k.extend(item)
    else:
        k.append(item)

unique_keywords = set(k)
print(unique_keywords)

{'deadlocks', 'denial of service', 'dead lock', 'vulnerable', 'stack overflow', 'underflow', 'racy', 'overflow', 'deadlock', 'race', 'leak', 'malicious', 'races'}


In [19]:
# Iterate over each element in df_java_sec
# and create a dict and sace it to a jsonl file
import json

with open('test_real.jsonl', 'w') as f:
    for index, row in df_java_sec.iterrows():
        path = ''
        for comment in comments:
            if comment['id'] == row['id']:
                path = comment['path']
                break
        data = {
            'id': row['id'],
            'project': f'{row["owner"]}/{row["name"]}',
            'file_path': path,
            'pr_id': row['pr_id'],
            'patch': row['diff'],
            'msg': row['body'],
            'url': row['url'],
            'lang': 'java',
        }
        f.write(json.dumps(data) + '\n')